# Final Model Selection & Saving

In this notebook, we will:

1. Load the feature-engineered dataset
2. Prepare the chronological train-test split
3. Train the candidate models
4. Compare their performance
5. Select the final model
6. Train the final model
7. Save the trained model
8. Save model metadata

The saved model will later be loaded by the FastAPI backend.

In [1]:
import pandas as pd
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib
import json

## 1. Load Feature-Engineered Dataset

We will use the feature-engineered dataset created in Notebook 4.

In [2]:
df = pd.read_csv(
    "../data/processed/PremierLeague25_26_features.csv"
)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

df.head()

,Date,HomeTeam,AwayTeam,HomeMatchesBefore,HomeGoalsForBefore,HomeGoalsAgainstBefore,HomePointsBefore,AwayMatchesBefore,AwayGoalsForBefore,AwayGoalsAgainstBefore,...,FTR,HomeAvgGoalsFor,HomeAvgGoalsAgainst,AwayAvgGoalsFor,AwayAvgGoalsAgainst,HomePointsPerMatch,AwayPointsPerMatch,AttackDifference,DefenseDifference,PointsDifference
0,2025-08-15,Liverpool,Bournemouth,0,0,0,0,0,0,0,...,H,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-08-16,Aston Villa,Newcastle,0,0,0,0,0,0,0,...,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-08-16,Brighton,Fulham,0,0,0,0,0,0,0,...,D,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-08-16,Sunderland,West Ham,0,0,0,0,0,0,0,...,H,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-08-16,Tottenham,Burnley,0,0,0,0,0,0,0,...,H,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
print("Dataset shape:", df.shape)

Dataset shape: (380, 21)


## 2. Define Features and Target

The target variable is FTR:

- H = Home Win
- D = Draw
- A = Away Win

The features represent historical team performance available before the match.

In [4]:
ml_features = [
    "HomeAvgGoalsFor",
    "HomeAvgGoalsAgainst",
    "AwayAvgGoalsFor",
    "AwayAvgGoalsAgainst",
    "HomePointsPerMatch",
    "AwayPointsPerMatch",
    "AttackDifference",
    "DefenseDifference",
    "PointsDifference"
]

X = df[ml_features]
y = df["FTR"]

In [5]:
print("Features:")
print(ml_features)

print("\nTarget distribution:")
print(y.value_counts())

Features:
['HomeAvgGoalsFor', 'HomeAvgGoalsAgainst', 'AwayAvgGoalsFor', 'AwayAvgGoalsAgainst', 'HomePointsPerMatch', 'AwayPointsPerMatch', 'AttackDifference', 'DefenseDifference', 'PointsDifference']

Target distribution:
FTR
H    162
A    114
D    104
Name: count, dtype: int64


## 3. Chronological Train-Test Split

We will use the first 80% of matches for training and the final 20% for testing.

The data is not randomly shuffled because football prediction is time-dependent.

In [6]:
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

X_train = train_df[ml_features]
X_test = test_df[ml_features]

y_train = train_df["FTR"]
y_test = test_df["FTR"]

In [7]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining period:")
print(train_df["Date"].min(), "to", train_df["Date"].max())

print("\nTesting period:")
print(test_df["Date"].min(), "to", test_df["Date"].max())

Training samples: 304
Testing samples: 76

Training period:
2025-08-15 00:00:00 to 2026-03-21 00:00:00

Testing period:
2026-03-21 00:00:00 to 2026-05-24 00:00:00


## 4. Train Candidate Models

We will train the following models:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting

The baseline model is only used as a reference and will not be considered as the final predictive model.

In [8]:
logistic_model = LogisticRegression(
    max_iter=1000
)

logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)

C:\Users\LENOVO\PycharmProjects\DataScience_Football\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


In [9]:
decision_tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

decision_tree.fit(X_train, y_train)

dt_predictions = decision_tree.predict(X_test)

In [10]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

random_forest.fit(X_train, y_train)

rf_predictions = random_forest.predict(X_test)

In [11]:
gradient_boosting = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gradient_boosting.fit(X_train, y_train)

gb_predictions = gradient_boosting.predict(X_test)

## 5. Compare Candidate Models

We will compare the models using:

- Accuracy
- Macro Precision
- Macro Recall
- Macro F1-score

Macro F1 is useful because it gives equal importance to the Home Win, Draw and Away Win classes.

In [12]:
model_predictions = {
    "Logistic Regression": logistic_predictions,
    "Decision Tree": dt_predictions,
    "Random Forest": rf_predictions,
    "Gradient Boosting": gb_predictions
}

comparison = []

for model_name, predictions in model_predictions.items():

    report = classification_report(
        y_test,
        predictions,
        output_dict=True,
        zero_division=0
    )

    comparison.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": report["macro avg"]["precision"],
        "Recall": report["macro avg"]["recall"],
        "F1": report["macro avg"]["f1-score"]
    })

comparison_df = pd.DataFrame(comparison)

comparison_df

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.552632,0.354167,0.433333,0.365000
1,Decision Tree,0.381579,0.264706,0.342593,0.294628
2,Random Forest,0.513158,0.385606,0.412963,0.372210
3,Gradient Boosting,0.473684,0.403704,0.407407,0.394480


In [13]:
comparison_df.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1
0,Gradient Boosting,0.473684,0.403704,0.407407,0.394480
1,Random Forest,0.513158,0.385606,0.412963,0.372210
2,Logistic Regression,0.552632,0.354167,0.433333,0.365000
3,Decision Tree,0.381579,0.264706,0.342593,0.294628


## 6. Select the Final Model

We will select the model with the strongest overall performance.

The primary comparison metric will be Macro F1-score, while accuracy, precision, recall and confusion matrices will also be considered.

The final model should not simply be selected because it has the highest accuracy.

In [14]:
best_model_name = comparison_df.loc[
    comparison_df["F1"].idxmax(),
    "Model"
]

best_model_name

'Gradient Boosting'

In [15]:
trained_models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree,
    "Random Forest": random_forest,
    "Gradient Boosting": gradient_boosting
}

final_model = trained_models[best_model_name]

final_model

GradientBoostingClassifier(learning_rate=0.05, random_state=42)

## 7. Final Model Evaluation

We will now evaluate the selected final model on the test set.

In [16]:
final_predictions = final_model.predict(X_test)

final_accuracy = accuracy_score(
    y_test,
    final_predictions
)

print("Final Model:", best_model_name)
print("Final Accuracy:", final_accuracy)

Final Model: Gradient Boosting
Final Accuracy: 0.47368421052631576


In [17]:
print(
    classification_report(
        y_test,
        final_predictions,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           A       0.33      0.35      0.34        20
           D       0.30      0.15      0.20        20
           H       0.58      0.72      0.64        36

    accuracy                           0.47        76
   macro avg       0.40      0.41      0.39        76
weighted avg       0.44      0.47      0.45        76



In [18]:
final_confusion_matrix = confusion_matrix(
    y_test,
    final_predictions,
    labels=["H", "D", "A"]
)

final_confusion_matrix

array([[26,  4,  6],
       [ 9,  3,  8],
       [10,  3,  7]])

## 8. Save the Final Model

We will save the trained model using Joblib.

The saved model can later be loaded by the FastAPI backend without retraining it every time the API starts.

In [19]:
model_path = "../models/final_model.joblib"

joblib.dump(
    final_model,
    model_path
)

print("Model saved to:", model_path)

Model saved to: ../models/final_model.joblib


In [20]:
loaded_model = joblib.load(
    model_path
)

print("Model loaded successfully!")

Model loaded successfully!


In [21]:
loaded_predictions = loaded_model.predict(X_test)

loaded_accuracy = accuracy_score(
    y_test,
    loaded_predictions
)

print("Loaded Model Accuracy:", loaded_accuracy)

Loaded Model Accuracy: 0.47368421052631576


In [22]:
print(
    "Accuracy matches:",
    final_accuracy == loaded_accuracy
)

Accuracy matches: True


## 9. Save Model Metadata

The model file contains the trained machine learning model.

We will separately save metadata describing:

- Model name
- Features used
- Target variable
- Class labels
- Test accuracy

In [23]:
metadata = {
    "model_name": best_model_name,
    "features": ml_features,
    "target": "FTR",
    "classes": ["H", "D", "A"],
    "accuracy": float(final_accuracy)
}

metadata_path = "../models/model_metadata.json"

with open(metadata_path, "w") as file:
    json.dump(
        metadata,
        file,
        indent=4
    )

print("Metadata saved to:", metadata_path)

Metadata saved to: ../models/model_metadata.json


In [24]:
with open(metadata_path, "r") as file:
    saved_metadata = json.load(file)

saved_metadata

{'model_name': 'Gradient Boosting',
 'features': ['HomeAvgGoalsFor',
  'HomeAvgGoalsAgainst',
  'AwayAvgGoalsFor',
  'AwayAvgGoalsAgainst',
  'HomePointsPerMatch',
  'AwayPointsPerMatch',
  'AttackDifference',
  'DefenseDifference',
  'PointsDifference'],
 'target': 'FTR',
 'classes': ['H', 'D', 'A'],
 'accuracy': 0.47368421052631576}

# Final Model Summary

The machine learning pipeline has now produced a final trained classification model.

The model:

- Predicts Home Win, Draw or Away Win
- Uses historical team-performance features
- Was trained using chronological historical data
- Was evaluated on later matches
- Was compared against multiple classification algorithms
- Has been saved using Joblib
- Has accompanying metadata

The saved model can now be integrated into the FastAPI backend.